# Web Chatbot Penelitian

Aplikasi ini dibangun dengan menggunakan arsitektur sederhana.
**Google Scholar** (melalui library `scholarly`) bertindak sebagai sumber data pencarian.
**Gemini API** bertindak sebagai otak LLM yang mensintesis jawaban dengan `temperature=0`.
**Streamlit** sebagai antarmuka penggunanya. **Ngrok** akan digunakan untuk mengekspos server lokal/Colab ke internet.

Source repo: [adiptamartulandi/chatbot-streamlit-demo](https://github.com/adiptamartulandi/chatbot-streamlit-demo)

## Setup: Install Library & Konfigurasi ngrok

### Step 1 — Install Streamlit dan pyngrok

In [ ]:
!pip install -q streamlit pyngrok google-genai scholarly

### Step 2 — Daftarkan ngrok Auth Token

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

### Fungsi Helper: Jalankan Streamlit + Buka Tunnel

In [ ]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

# Chatbot dengan Gemini & Scholarly

Sekarang kita masuk ke bagian utama — membangun chatbot berbasis LLM menggunakan Streamlit + Gemini API.
Sumber: `streamlit_chat_app.py` dari repo demo.


In [ ]:
%%writefile streamlit_chat_app.py
# Import library yang dibutuhkan
import streamlit as st          # framework web app
from google import genai         # SDK Gemini dari Google
from scholarly import scholarly   # Library untuk mencari paper di Google Scholar
import json                      # Untuk memproses komunikasi terstruktur dengan Gemini

# ── 1. Konfigurasi Halaman ───────────────────────────────────────────────────
st.set_page_config(
    page_title="Gemini Research Assistant",
    page_icon="🔬",
    layout="centered"
)
st.title("Gemini Academic & Research Chatbot")
st.caption("Chatbot pintar bertenaga Google Gemini Flash & Real-time Google Scholar Search")

# ── 2. Sidebar: Pengaturan App ───────────────────────────────────────────────
with st.sidebar:
    st.subheader("Pengaturan API")
    # Kotak input untuk API key (type="password" untuk menyembunyikan teks)
    google_api_key = st.text_input("Google AI API Key", type="password")

    st.markdown("---")
    st.subheader("⚙️ Fitur Asisten Penelitian")

    # Checkbox aktifasi Google Scholar
    scholar_mode = st.checkbox(
        "Integrasi Google Scholar",
        value=True,
        help="Jika aktif, bot akan mencari literatur ilmiah nyata sebelum menjawab pertanyaan akademikmu."
    )

    # Slider batas maksimal paper yang diambil sekali pencarian
    max_papers = st.slider(
        "Jumlah Referensi Maksimal",
        min_value=1,
        max_value=5,
        value=3,
        help="Makin banyak jumlah paper, proses penarikan data memerlukan waktu sedikit lebih lama."
    )

    st.markdown("---")
    # Tombol untuk mereset percakapan
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

    st.caption("""**Panduan Mencoba App:**
1. Masukkan Google AI API Key kamu di atas.
2. Centang **Integrasi Google Scholar** untuk mengaktifkan fitur pencarian jurnal otomatis.
3. Tanyakan hal seputar riset, ide judul, teori, atau minta analisis tren riset ilmiah.
4. Bot akan otomatis mengekstrak kata kunci dan menyajikan data referensi nyata.""")

# ── 3. Validasi API Key ──────────────────────────────────────────────────────
if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai menggunakan asisten penelitian.", icon="🗝️")
    st.stop()

# ── 4. Inisialisasi Gemini Client ────────────────────────────────────────────
if ("genai_client" not in st.session_state) or (
    getattr(st.session_state, "_last_key", None) != google_api_key
):
    try:
        st.session_state.genai_client = genai.Client(api_key=google_api_key)
        st.session_state._last_key = google_api_key

        # Hapus session lama jika berganti kunci API
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

# ── 5. Fungsi Pembantu: Google Scholar Search ────────────────────────────────
def cari_literatur_scholar(query: str, limit: int):
    """
    Mengambil data dari Google Scholar menggunakan library scholarly.
    Mengembalikan daftar paper berupa dictionary berisi metadata esensial.
    """
    try:
        search_query = scholarly.search_pubs(query)
        daftar_paper = []
        for _ in range(limit):
            try:
                paper = next(search_query)
                bib = paper.get('bib', {})

                # Ekstraksi informasi dasar literatur ilmiah
                item = {
                    "judul": bib.get('title', 'Tidak ada judul'),
                    "penulis": bib.get('author', ['Tidak diketahui']),
                    "tahun": bib.get('pub_year', 'Tidak tersedia'),
                    "jurnal": bib.get('venue', 'Tidak tersedia'),
                    "abstrak": bib.get('abstract', 'Abstrak tidak ditampilkan secara publik oleh Google Scholar.'),
                    "url": paper.get('pub_url', '#')
                }
                daftar_paper.append(item)
            except StopIteration:
                break
        return daftar_paper
    except Exception as e:
        return {"error": str(e)}

# ── 6. Inisialisasi Chat Session & Riwayat Pesan ────────────────────────────
if "chat" not in st.session_state:
    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash",
        config={"temperature": 0.2} # Temperature sedikit dinaikkan agar ulasan akademis lebih luwes
    )

if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 7. Tombol Reset ──────────────────────────────────────────────────────────
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    st.rerun()

# ── 8. Tampilkan Riwayat Percakapan ─────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        # Tampilkan referensi yang pernah digunakan jika tersimpan pada riwayat asisten
        if msg["role"] == "assistant" and "references" in msg and msg["references"]:
            with st.expander("📚 Referensi Ilmiah yang digunakan pada sesi ini:"):
                for idx, ref in enumerate(msg["references"]):
                    st.markdown(f"**{idx+1}. {ref['judul']}** ({ref['tahun']})")
                    st.markdown(f"*Penulis:* {', '.join(ref['penulis'])} | *Jurnal:* {ref['jurnal']}")
                    st.markdown(f"[Buka Publikasi]({ref['url']})")
                    st.markdown(f"**Abstrak:** {ref['abstrak']}")
                    st.markdown("---")
        st.markdown(msg["content"])

# ── 9. Input & Respons Utama Asisten ─────────────────────────────────────────
prompt = st.chat_input("Ketik pertanyaan penelitian atau topik risetmu di sini...")

if prompt:
    # Tampilkan & simpan prompt asli user
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    references_found = []
    augmented_context = ""

    # Proses pencarian pintar berbasis Google Scholar
    if scholar_mode:
        # Menggunakan st.status untuk memperlihatkan progress pencarian latar belakang secara elegan
        with st.status("🔍 Menganalisis kebutuhan literatur akademik...", expanded=True) as status:

            # Meminta Gemini mendeteksi apakah prompt berupa obrolan biasa atau riset yang memerlukan paper
            check_prompt = f"""Pesan user: "{prompt}"
Analisislah apakah pesan ini membutuhkan referensi jurnal/paper ilmiah nyata.
Berikan output berupa JSON murni dengan format berikut:
{{
  "butuh_pencarian": true/false,
  "query_pencarian": "kata kunci spesifik dalam bahasa Inggris/Indonesia untuk dimasukkan ke Google Scholar"
}}"""
            try:
                check_response = st.session_state.genai_client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=check_prompt,
                    config={"response_mime_type": "application/json"}
                )
                res_data = json.loads(check_response.text)
                butuh_pencarian = res_data.get("butuh_pencarian", False)
                query_scholar = res_data.get("query_pencarian", prompt)
            except Exception:
                # Fallback jika kegagalan struktural JSON terjadi
                butuh_pencarian = True
                query_scholar = prompt

            if butuh_pencarian:
                status.update(label=f"📚 Mencari Google Scholar untuk: '{query_scholar}'...")
                papers = cari_literatur_scholar(query_scholar, max_papers)

                if isinstance(papers, dict) and "error" in papers:
                    st.error(f"Gagal memanggil Google Scholar: {papers['error']}")
                elif not papers:
                    st.warning("Tidak ditemukan paper ilmiah yang cocok di Google Scholar.")
                else:
                    references_found = papers

                    # Bangun konteks berbasis literatur nyata untuk disuntikkan ke Gemini
                    augmented_context = "Berikut adalah pustaka/literatur ilmiah riil dari Google Scholar:\n\n"
                    for idx, p in enumerate(papers):
                        augmented_context += (
                            f"Paper #{idx+1}:\n"
                            f"Judul: {p['judul']}\n"
                            f"Penulis: {', '.join(p['penulis'])}\n"
                            f"Tahun: {p['tahun']}\n"
                            f"Jurnal: {p['jurnal']}\n"
                            f"Abstrak: {p['abstrak']}\n\n"
                        )
            else:
                st.write("Pertanyaan bersifat umum, menjawab tanpa memerlukan pencarian database Scholar.")

            status.update(label="Analisis literatur selesai!", state="complete", expanded=False)

    # Menampilkan respons jawaban asisten akademik
    with st.chat_message("assistant"):
        # Jika ada referensi hasil pencarian scholar, render expander-nya di UI chat sekarang
        if references_found:
            with st.expander("📚 Referensi Google Scholar yang ditemukan:"):
                for idx, ref in enumerate(references_found):
                    st.markdown(f"**{idx+1}. {ref['judul']}** ({ref['tahun']})")
                    st.markdown(f"*Penulis:* {', '.join(ref['penulis'])} | *Jurnal:* {ref['jurnal']}")
                    st.markdown(f"[Buka Publikasi]({ref['url']})")
                    st.markdown(f"**Abstrak:** {ref['abstrak']}")
                    st.markdown("---")

        try:
            # Jika konteks terkumpul, lakukan teknik grounding (RAG) pada prompt final
            if augmented_context:
                final_prompt = f"""Konteks Pustaka Ilmiah:\n{augmented_context}\n\nInstruksi: Jawablah pertanyaan user berikut dengan gaya bahasa akademis, obyektif, dan komprehensif berlandaskan data Konteks Pustaka Ilmiah di atas. Lakukan sitasi tekstual (misal: Nama Penulis, Tahun) dalam ulasan jawabanmu jika merujuk pada poin paper tersebut.\n\nPertanyaan User: {prompt}"""
            else:
                final_prompt = prompt

            # Kirim pesan ke chat session agar Gemini mengingat histori alur riset sebelumnya
            response = st.session_state.chat.send_message(final_prompt)
            answer = response.text if hasattr(response, "text") else str(response)

        except Exception as e:
            answer = f"Terjadi error saat merangkum jawaban: {e}"

        st.markdown(answer)

    # Simpan respons asisten berserta metadata referensinya ke dalam session_state agar persisten saat re-run halaman
    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "references": references_found
    })

Jalankan chatbot-nya:

In [ ]:
proc = run_streamlit("streamlit_chat_app.py")

## Menghentikan App

Jalankan cell di bawah untuk menghentikan Streamlit dan menutup tunnel ngrok.

In [ ]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")